In [1]:
!pip install pbr importRosbag expelliarmus --no-deps -q
!pip install tonic --no-deps -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.9/131.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 3.9 MB/s eta 0:00:00


In [2]:
#pip install tonic


In [3]:
"""
hhpc_stat_analysis_src.py
==========================
Statistical-analysis harness built on hhpc_base.py: HH-PC vs LIF-PC,
paired t-test across seeds, run over MNIST / FashionMNIST / Caltech
(Faces_easy vs Motorbikes) / N-MNIST.

Extends the base module with:
    - LIFNeuron (ported, matched to HHNeuron's structure/conventions)
    - PCSNNet gains a `neuron_type` switch ("hh" | "lif") instead of
      forking into two classes -- pc_infer/pc_learn/train_step stay
      neuron-agnostic and untouched, per the base module's own docstring.
    - forward_proxies gains an "event_direct" branch for pre-binned
      event-camera frames (N-MNIST), alongside the existing poisson /
      latency_first branches.
    - eval_epoch / spike_rate_epoch / train gain an optional
      batch_preprocess_fn (needed because N-MNIST batches are
      (B, T, D) event frames, not flat images).
    - Caltech and N-MNIST loaders (MNIST/FashionMNIST reuse the base
      get_loaders unchanged).
    - run_statistical_analysis: trains HH-PC and LIF-PC per seed,
      reports mean/std and a paired t-test on test accuracy and F1.
"""

import os
import time
from dataclasses import dataclass
from typing import List, Optional, Dict, Any, Tuple, Callable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from scipy import stats
from tqdm import tqdm


# ─────────────────────────────────────────────────────────────────────────
# 0. UTILITIES  (unchanged from hhpc_base.py, + np seed for parity)
# ─────────────────────────────────────────────────────────────────────────

def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def default_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def one_hot(y: torch.Tensor, num_classes: int) -> torch.Tensor:
    return F.one_hot(y.long(), num_classes=num_classes).float()


# ─────────────────────────────────────────────────────────────────────────
# 1. DATA LOADERS
# ─────────────────────────────────────────────────────────────────────────

def get_loaders(
    dataset_name: str = "MNIST",
    batch_size: int = 128,
    root: str = "./data",
    device: str = "cpu",
    val_ratio: float = 0.1,
    seed: int = 42,
):
    """MNIST / FashionMNIST / KMNIST loader -- unchanged from hhpc_base.py."""
    tfm = transforms.Compose([transforms.ToTensor()])
    ds = dataset_name.upper()

    if ds == "KMNIST":
        train_full = datasets.KMNIST(root=root, train=True, download=True, transform=tfm)
        test_ds = datasets.KMNIST(root=root, train=False, download=True, transform=tfm)
    elif ds in ("FMNIST", "FASHIONMNIST"):
        train_full = datasets.FashionMNIST(root=root, train=True, download=True, transform=tfm)
        test_ds = datasets.FashionMNIST(root=root, train=False, download=True, transform=tfm)
    else:
        train_full = datasets.MNIST(root=root, train=True, download=True, transform=tfm)
        test_ds = datasets.MNIST(root=root, train=False, download=True, transform=tfm)

    n_total = len(train_full)
    n_val = max(1, int(round(val_ratio * n_total)))
    n_train = n_total - n_val
    g = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(train_full, [n_train, n_val], generator=g)

    use_cuda = device.startswith("cuda") and torch.cuda.is_available()
    kw = dict(num_workers=2 if use_cuda else 0, pin_memory=use_cuda)

    return (
        DataLoader(train_ds, batch_size, shuffle=True, **kw),
        DataLoader(val_ds, batch_size, shuffle=False, **kw),
        DataLoader(test_ds, batch_size, shuffle=False, **kw),
    )


def get_caltech_loaders(batch_size: int = 128, root: str = "./data", device: str = "cpu", seed: int = 42):
    """Binary Caltech101 subset: Faces_easy (0) vs Motorbikes (1), 32x32 grayscale.
    Flattening happens at batch-preprocess time (same convention as get_loaders),
    not in the transform."""
    tf = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
    ])
    full_ds = datasets.Caltech101(root=root, download=True, transform=tf)

    cat_to_idx = {c: i for i, c in enumerate(full_ds.categories)}
    face_idx = cat_to_idx["Faces_easy"]
    moto_idx = cat_to_idx["Motorbikes"]
    indices = [i for i, lbl in enumerate(full_ds.y) if lbl in (face_idx, moto_idx)]
    label_map = {face_idx: 0, moto_idx: 1}

    class _BinarySubset(torch.utils.data.Dataset):
        def __init__(self, ds, idxs, lmap):
            self.ds, self.idxs, self.lmap = ds, idxs, lmap

        def __len__(self):
            return len(self.idxs)

        def __getitem__(self, i):
            x, y = self.ds[self.idxs[i]]
            return x, self.lmap[y]

    binary_ds = _BinarySubset(full_ds, indices, label_map)
    n = len(binary_ds)
    n_test = max(1, int(n * 0.15))
    n_val = max(1, int(n * 0.10))
    n_train = n - n_val - n_test
    g = torch.Generator().manual_seed(seed)
    train_ds, val_ds, test_ds = random_split(binary_ds, [n_train, n_val, n_test], generator=g)

    use_cuda = device.startswith("cuda") and torch.cuda.is_available()
    kw = dict(num_workers=2 if use_cuda else 0, pin_memory=use_cuda)

    return (
        DataLoader(train_ds, batch_size, shuffle=True, **kw),
        DataLoader(val_ds, batch_size, shuffle=False, **kw),
        DataLoader(test_ds, batch_size, shuffle=False, **kw),
    )


def get_nmnist_loaders(batch_size: int = 128, root: str = "./data", device: str = "cpu",
                        steps_spk: int = 25, val_ratio: float = 0.1, seed: int = 42):
    """N-MNIST loader via tonic. Requires: pip install tonic.
    Each sample is pre-binned into (T, 2, 34, 34) frames; polarity merging
    into (T, 1156) happens in nmnist_preprocess at batch time."""
    try:
        import tonic
        import tonic.transforms as TT
    except ImportError as exc:
        raise ImportError("N-MNIST needs tonic: pip install tonic") from exc

    sensor_size = tonic.datasets.NMNIST.sensor_size  # (34, 34, 2)
    frame_tf = TT.ToFrame(sensor_size=sensor_size, n_time_bins=steps_spk)

    train_full = tonic.datasets.NMNIST(save_to=root, train=True, transform=frame_tf)
    test_ds = tonic.datasets.NMNIST(save_to=root, train=False, transform=frame_tf)

    n_total = len(train_full)
    n_val = max(1, int(round(val_ratio * n_total)))
    n_train = n_total - n_val
    g = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(train_full, [n_train, n_val], generator=g)

    def collate(batch):
        xs, ys = zip(*batch)
        return torch.from_numpy(np.stack(xs)), torch.tensor(ys)

    kw = dict(num_workers=0, collate_fn=collate)
    return (
        DataLoader(train_ds, batch_size, shuffle=True, **kw),
        DataLoader(val_ds, batch_size, shuffle=False, **kw),
        DataLoader(test_ds, batch_size, shuffle=False, **kw),
    )


def nmnist_preprocess(x_raw: torch.Tensor, device: str) -> torch.Tensor:
    """(B, T, 2, 34, 34) -> (B, T, 1156), polarities summed and clamped to {0,1}."""
    x = x_raw.to(device).float()
    x = x.sum(dim=2)                        # merge polarities -> (B, T, 34, 34)
    x = x.view(x.size(0), x.size(1), -1)    # -> (B, T, 1156)
    return x.clamp(0, 1)


def default_batch_preprocess(x_raw: torch.Tensor, device: str) -> torch.Tensor:
    """Flatten-to-vector, used for MNIST / FashionMNIST / Caltech."""
    return x_raw.to(device).view(x_raw.size(0), -1)


# ─────────────────────────────────────────────────────────────────────────
# 2. HODGKIN-HUXLEY NEURON  (verbatim from hhpc_base.py)
# ─────────────────────────────────────────────────────────────────────────

class HHNeuron(nn.Module):
    class Gate:
        def __init__(self, B, N, device):
            self.alpha = torch.zeros(B, N, device=device)
            self.beta = torch.zeros(B, N, device=device)
            self.state = torch.zeros(B, N, device=device)

        def update(self, dt):
            s = self.state + dt * (self.alpha * (1 - self.state) - self.beta * self.state)
            return s.clamp(0.0, 1.0)

        def set_inf(self):
            self.state = self.alpha / (self.alpha + self.beta + 1e-8)

    def __init__(self, N, dt=0.03, device="cpu", thr=0.8, reset=0.0, tau_ref=2.0):
        super().__init__()
        self.N, self.dt, self.device = N, float(dt), device
        self.thr, self.reset = float(thr), float(reset)
        self.refr_steps = max(1, int(round(tau_ref / dt)))
        for name, val in [("ENa", 115.), ("EK", -12.), ("Eleak", 10.6),
                           ("gNa", 120.), ("gK", 36.), ("gLeak", 0.3), ("Cm", 1.)]:
            self.register_buffer(name, torch.tensor(val))
        self.reset_states(1)

    def reset_states(self, B):
        dev = self.device
        self.B = B
        self.Vm = torch.zeros(B, self.N, device=dev)
        self.m = HHNeuron.Gate(B, self.N, dev)
        self.n = HHNeuron.Gate(B, self.N, dev)
        self.h = HHNeuron.Gate(B, self.N, dev)
        self._update_gates(self.Vm)
        self.m.set_inf(); self.n.set_inf(); self.h.set_inf()
        self.refr = torch.zeros(B, self.N, device=dev)

    def _update_gates(self, V):
        V = V.clamp(-100., 100.)
        self.n.alpha = 0.01 * (10 - V) / (torch.exp((10 - V) / 10) - 1 + 1e-8)
        self.n.beta = 0.125 * torch.exp(-V / 80.)
        self.m.alpha = 0.1 * (25 - V) / (torch.exp((25 - V) / 10) - 1 + 1e-8)
        self.m.beta = 4. * torch.exp(-V / 18.)
        self.h.alpha = 0.07 * torch.exp(-V / 20.)
        self.h.beta = 1. / (torch.exp((30 - V) / 10) + 1.)

    def forward(self, I):
        if I.shape[0] != self.B:
            self.reset_states(I.shape[0])
        self._update_gates(self.Vm)
        m = self.m.update(self.dt); n = self.n.update(self.dt); h = self.h.update(self.dt)
        INa = m ** 3 * self.gNa * h * (self.Vm - self.ENa)
        IK = n ** 4 * self.gK * (self.Vm - self.EK)
        IL = self.gLeak * (self.Vm - self.Eleak)
        dV = (I - INa - IK - IL) / self.Cm
        Vn = self.Vm + self.dt * dV
        Vn = torch.tanh(Vn / 30.) * 30.
        can = (self.refr <= 0)
        spk = ((Vn >= self.thr) & can).float()
        self.Vm = torch.where(spk.bool(), torch.full_like(Vn, self.reset), Vn)
        self.m.state = m; self.n.state = n; self.h.state = h
        self.refr = torch.where(spk.bool(),
                                 torch.full_like(self.refr, float(self.refr_steps)),
                                 (self.refr - 1.).clamp(min=0.))
        return spk, self.Vm


# ─────────────────────────────────────────────────────────────────────────
# 2b. LIF NEURON  (ported, matched to HHNeuron's own conventions above:
#     refr_steps computed once in __init__, same reset_states/forward shape)
# ─────────────────────────────────────────────────────────────────────────

class LIFNeuron(nn.Module):
    def __init__(self, N, dt=0.03, device="cpu", thr=0.8, reset=0.0, tau=0.3, tau_ref=2.0):
        super().__init__()
        self.N, self.dt, self.device = N, float(dt), device
        self.thr, self.reset, self.tau = float(thr), float(reset), float(tau)
        self.refr_steps = max(1, int(round(tau_ref / dt)))
        self.reset_states(1)

    def reset_states(self, B):
        dev = self.device
        self.B = B
        self.Vm = torch.zeros(B, self.N, device=dev)
        self.refr = torch.zeros(B, self.N, device=dev)

    def forward(self, I):
        if I.shape[0] != self.B:
            self.reset_states(I.shape[0])
        can = (self.refr <= 0)
        dV = (-self.Vm + I) * (self.dt / max(self.tau, 1e-6))
        Vn = self.Vm + dV
        spk = ((Vn >= self.thr) & can).float()
        self.Vm = torch.where(spk.bool(), torch.full_like(Vn, self.reset), Vn)
        self.refr = torch.where(spk.bool(),
                                 torch.full_like(self.refr, float(self.refr_steps)),
                                 (self.refr - 1.).clamp(min=0.))
        return spk, self.Vm


# ─────────────────────────────────────────────────────────────────────────
# 3. PC ACTIVATION  (unchanged)
# ─────────────────────────────────────────────────────────────────────────

def make_pc_activation(name: str):
    name = name.lower()
    if name == "sigmoid":
        return torch.sigmoid, lambda z: torch.sigmoid(z) * (1 - torch.sigmoid(z))
    if name == "tanh":
        return torch.tanh, lambda z: 1 - torch.tanh(z) ** 2
    def f(z): return z.clamp(0., 1.)
    def fp(z): return ((z > 0.) & (z < 1.)).float()
    return f, fp


# ─────────────────────────────────────────────────────────────────────────
# 4. PC-SNN NETWORK  (base PCSNNet + neuron_type switch + event_direct)
# ─────────────────────────────────────────────────────────────────────────

class PCSNNet(nn.Module):
    """Two-layer PC network: input -> hidden (spiking) -> output (spiking).
    neuron_type = "hh" | "lif". input_encoding = "poisson" | "latency_first" | "event_direct".
    pc_infer / pc_learn / train_step are neuron-agnostic and unchanged from hhpc_base.py."""

    def __init__(
        self,
        layer_sizes: List[int],
        dt: float = 0.03,
        device: str = "cpu",
        neuron_type: str = "hh",
        current_gain: float = 30.0,
        I_bias: float = 2.0,
        thr: float = 0.8,
        lif_tau: float = 0.3,
        pc_activation: str = "relu",
        lr: float = 2e-4,
        weight_decay: float = 1e-4,
        input_encoding: str = "poisson",
        poisson_scale: float = 1.0,
    ):
        super().__init__()
        assert len(layer_sizes) >= 2
        assert neuron_type in ("hh", "lif")
        assert input_encoding.lower() in ("poisson", "latency_first", "event_direct")
        self.device = torch.device(device)
        self.sizes = layer_sizes
        self.dt = float(dt)
        self.neuron_type = neuron_type
        self.current_gain = float(current_gain)
        self.I_bias = float(I_bias)
        self.input_encoding = input_encoding.lower()
        self.poisson_scale = float(poisson_scale)
        self.f, self.fprime = make_pc_activation(pc_activation)
        self.L = len(layer_sizes) - 1
        self.S = self.L  # both layers spike

        self.syn = nn.ModuleList([
            nn.Linear(layer_sizes[i], layer_sizes[i + 1], bias=True)
            for i in range(self.L)
        ])
        for lin in self.syn:
            nn.init.xavier_uniform_(lin.weight, gain=0.5)
            nn.init.zeros_(lin.bias)

        if neuron_type == "hh":
            self.cells = nn.ModuleList([
                HHNeuron(layer_sizes[i + 1], dt=dt, device=device, thr=thr)
                for i in range(self.S)
            ])
        else:
            self.cells = nn.ModuleList([
                LIFNeuron(layer_sizes[i + 1], dt=dt, device=device, thr=thr, tau=lif_tau)
                for i in range(self.S)
            ])

        self.opt = torch.optim.Adam(self.syn.parameters(), lr=lr, weight_decay=weight_decay)
        self._last_spike_sums: Optional[List[torch.Tensor]] = None

    @torch.no_grad()
    def _encode_latency(self, x0: torch.Tensor, steps: int) -> torch.Tensor:
        lat = steps * (1.0 - x0.clamp(0, 1))
        return lat.clamp(0., float(steps))

    @torch.no_grad()
    def _build_spike_train(self, latencies: torch.Tensor, steps: int) -> torch.Tensor:
        tgrid = torch.arange(1, steps + 1, device=self.device).view(1, 1, -1)
        return (latencies.unsqueeze(-1) <= tgrid).float()

    @torch.no_grad()
    def forward_proxies(self, x_in: torch.Tensor, steps_spk: int) -> List[torch.Tensor]:
        if self.input_encoding == "event_direct":
            assert x_in.dim() == 3, f"event_direct expects (B, T, D), got {tuple(x_in.shape)}"
            x_event = x_in.to(self.device).float().clamp(0, 1)
            steps_use = min(x_event.size(1), steps_spk)
            x0 = x_event[:, :steps_use, :].mean(dim=1).clamp(0, 1)
        else:
            x0 = x_in.to(self.device).clamp(0, 1)
            steps_use = steps_spk
        B = x0.size(0)

        for cell in self.cells:
            cell.reset_states(B)

        if self.input_encoding == "latency_first":
            lat = self._encode_latency(x0, steps_use)
            spk_train = self._build_spike_train(lat, steps_use)
            fired = torch.zeros_like(x0, dtype=torch.bool)
        elif self.input_encoding == "poisson":
            p = (x0 * self.poisson_scale).clamp(0, 1)
            spk_train = (torch.rand(B, x0.size(1), steps_use,
                                     device=self.device) < p.unsqueeze(-1)).float()

        spike_sums = [torch.zeros(B, self.sizes[i + 1], device=self.device)
                      for i in range(self.S)]

        for t in range(steps_use):
            if self.input_encoding == "event_direct":
                inp = x_event[:, t, :]
            elif self.input_encoding == "poisson":
                inp = spk_train[:, :, t]
            else:
                inp = (spk_train[:, :, t] * (~fired)).float()
                fired.logical_or_(inp.bool())

            r = inp
            for i in range(self.S):
                h = F.linear(r, self.syn[i].weight, self.syn[i].bias)
                I = h * self.current_gain + self.I_bias
                spk, _ = self.cells[i](I)
                spike_sums[i] += spk
                r = spk

        self._last_spike_sums = spike_sums
        proxies = [x0] + [(ss / float(steps_use)).clamp(0, 1) for ss in spike_sums]
        return proxies

    def last_spike_sums(self):
        return self._last_spike_sums

    # ── PC inference / learning (neuron-agnostic, unchanged from base) ──

    def pc_infer(self, x_init, y_target=None, T_infer=50, eta_x=0.05, clamp_output=True):
        L = self.L
        x = [xi.clone().detach().to(self.device) for xi in x_init]
        x[0] = x[0].clamp(0, 1)
        if clamp_output and y_target is not None:
            x[L] = y_target.clone().detach().to(self.device).clamp(0, 1)

        z_cache = [None] * L
        for _ in range(T_infer):
            e = [None] * (L + 1)
            e[0] = torch.zeros_like(x[0])
            for l in range(1, L + 1):
                idx = l - 1
                z = F.linear(x[l - 1], self.syn[idx].weight, self.syn[idx].bias)
                z_cache[idx] = z
                e[l] = x[l] - self.f(z)
            for l in range(1, L):
                fb = (e[l + 1] * self.fprime(z_cache[l])) @ self.syn[l].weight
                x[l] = (x[l] - eta_x * (e[l] - fb)).clamp_(0, 1)
            if not clamp_output:
                x[L] = (x[L] - eta_x * e[L]).clamp_(0, 1)

        energy = 0.
        with torch.no_grad():
            for l in range(1, L + 1):
                idx = l - 1
                z = F.linear(x[l - 1], self.syn[idx].weight, self.syn[idx].bias)
                el = x[l] - self.f(z)
                energy += 0.5 * (el ** 2).mean().item()
        return x, e, z_cache, energy

    def pc_learn(self, x, e, z_cache):
        B = x[0].shape[0]
        self.opt.zero_grad()
        for idx in range(self.L):
            local = e[idx + 1] * self.fprime(z_cache[idx])
            self.syn[idx].weight.grad = -(local.T @ x[idx]) / B
            self.syn[idx].bias.grad = -local.mean(0)
        torch.nn.utils.clip_grad_norm_(self.syn.parameters(), 1.0)
        self.opt.step()

    def train_step(self, x_in, y_target, steps_spk, T_infer, eta_x):
        proxies = self.forward_proxies(x_in, steps_spk)
        x, e, z_cache, energy = self.pc_infer(proxies, y_target, T_infer, eta_x, True)
        self.pc_learn(x, e, z_cache)
        return energy, proxies


# ─────────────────────────────────────────────────────────────────────────
# 5. METRICS  (unchanged from hhpc_base.py)
# ─────────────────────────────────────────────────────────────────────────

class MetricAccumulator:
    def __init__(self, C, device="cpu"):
        self.C = C; self.device = device; self.reset()

    def reset(self):
        self.tp = torch.zeros(self.C, dtype=torch.long)
        self.fp = torch.zeros(self.C, dtype=torch.long)
        self.fn = torch.zeros(self.C, dtype=torch.long)
        self.correct = self.total = 0

    @torch.no_grad()
    def update(self, pred, y):
        pred = pred.view(-1).long(); y = y.view(-1).long()
        self.total += y.numel()
        self.correct += int((pred == y).sum())
        tp = torch.bincount(pred[pred == y], minlength=self.C)
        pc = torch.bincount(pred, minlength=self.C)
        tc = torch.bincount(y, minlength=self.C)
        self.tp += tp; self.fp += pc - tp; self.fn += tc - tp

    def compute(self, eps=1e-8):
        tp = self.tp.float(); fp = self.fp.float(); fn = self.fn.float()
        P = (tp / (tp + fp + eps)).mean().item()
        R = (tp / (tp + fn + eps)).mean().item()
        F1 = (2 * tp / (2 * tp + fp + fn + eps)).mean().item()
        return {"acc": self.correct / max(self.total, 1), "precision": P, "recall": R, "f1": F1}


# ─────────────────────────────────────────────────────────────────────────
# 6. EVALUATION  (base eval_epoch / spike_rate_epoch + batch_preprocess_fn)
# ─────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def spike_rate_epoch(model, loader, device, steps_spk, eval_seed=1234, batch_preprocess_fn=None):
    model.eval()
    prep = batch_preprocess_fn or default_batch_preprocess
    S = model.S
    total_spk = [0.] * S
    total_den = [0.] * S
    n_samples = 0

    with torch.random.fork_rng():
        torch.manual_seed(eval_seed)
        for x, _ in loader:
            x = prep(x, device)
            B = x.size(0)
            model.forward_proxies(x, steps_spk)
            ss = model.last_spike_sums()
            if ss is None:
                continue
            for li in range(S):
                total_spk[li] += float(ss[li].sum())
                total_den[li] += float(B * ss[li].shape[1] * steps_spk)
            n_samples += B

    per_layer = [total_spk[li] / max(total_den[li], 1.) for li in range(S)]
    total = sum(total_spk) / max(sum(total_den), 1.)
    sps = sum(total_spk) / max(n_samples, 1)
    return {"per_layer": per_layer, "total": total, "spikes_per_sample": sps}


@torch.no_grad()
def eval_epoch(model, loader, device, steps_spk, T_infer, eta_x, eval_mode="pc",
                eval_seed=1234, batch_preprocess_fn=None):
    model.eval()
    prep = batch_preprocess_fn or default_batch_preprocess
    C = model.sizes[-1]
    ff_acc = MetricAccumulator(C)
    pc_acc = MetricAccumulator(C)
    total_e, total = 0., 0

    with torch.random.fork_rng():
        torch.manual_seed(eval_seed)
        for x, y in loader:
            x = prep(x, device)
            y = y.to(device)
            B = x.size(0)
            proxies = model.forward_proxies(x, steps_spk)

            ff_pred = proxies[-1].argmax(1)
            ff_acc.update(ff_pred.cpu(), y.cpu())

            if eval_mode == "pc":
                xs, _, _, _ = model.pc_infer(proxies, None, T_infer, eta_x, False)
                pc_pred = xs[-1].argmax(1)
            else:
                pc_pred = ff_pred
            pc_acc.update(pc_pred.cpu(), y.cpu())

            y_oh = one_hot(y, C)
            _, _, _, e = model.pc_infer(proxies, y_oh, T_infer, eta_x, True)
            total_e += e * B; total += B

    ff_m = ff_acc.compute(); pc_m = pc_acc.compute()
    return {
        "ff_acc": ff_m["acc"], "ff_f1": ff_m["f1"],
        "pc_acc": pc_m["acc"], "pc_f1": pc_m["f1"],
        "pc_precision": pc_m["precision"], "pc_recall": pc_m["recall"],
        "pc_energy": total_e / max(total, 1),
    }


# ─────────────────────────────────────────────────────────────────────────
# 7. TRAINING LOOP  (base train() + batch_preprocess_fn + progress bar)
# ─────────────────────────────────────────────────────────────────────────

class EarlyStopper:
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.best = None; self.bad = 0; self.best_epoch = 0

    def step(self, val, epoch):
        if self.best is None or val > self.best + self.min_delta:
            self.best = val; self.bad = 0; self.best_epoch = epoch
            return False, True
        self.bad += 1
        return (self.bad >= self.patience), False


def train(model, train_loader, val_loader, device,
          steps_spk, T_infer_train, T_infer_eval,
          eta_x, epochs, eval_mode, eval_seed,
          patience, ckpt_path, verbose=False,
          batch_preprocess_fn=None, desc=""):

    prep = batch_preprocess_fn or default_batch_preprocess
    stopper = EarlyStopper(patience=patience)
    best_val_acc = 0.

    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        pbar = tqdm(train_loader, desc=f"{desc} ep{epoch}/{epochs}", ncols=100, disable=not verbose)
        for x, y in pbar:
            x = prep(x, device)
            y = y.to(device)
            energy, _ = model.train_step(x, one_hot(y, model.sizes[-1]), steps_spk, T_infer_train, eta_x)
            pbar.set_postfix({"E": f"{energy:.4f}"})

        val_stats = eval_epoch(model, val_loader, device, steps_spk, T_infer_eval, eta_x,
                                eval_mode, eval_seed, batch_preprocess_fn)
        monitor = val_stats["pc_acc"] if eval_mode == "pc" else val_stats["ff_acc"]

        stop, improved = stopper.step(monitor, epoch)
        if improved:
            torch.save(model.state_dict(), ckpt_path)
            best_val_acc = monitor
        if verbose:
            print(f"  [{desc}] epoch {epoch:02d} | val_acc={monitor:.4f} | "
                  f"{time.time() - t0:.1f}s{' *' if improved else ''}")
        if stop:
            if verbose:
                print(f"  [{desc}] early stop at epoch {epoch}")
            break

    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return best_val_acc


# ─────────────────────────────────────────────────────────────────────────
# 8. DATASET REGISTRY
# ─────────────────────────────────────────────────────────────────────────

@dataclass
class DatasetSpec:
    layer_sizes: Tuple[int, int, int]
    input_encoding: str
    steps_spk: int
    loader_fn: Callable[[int, str, int], Tuple[DataLoader, DataLoader, DataLoader]]
    batch_preprocess_fn: Optional[Callable] = None


DATASETS: Dict[str, DatasetSpec] = {
    "MNIST": DatasetSpec(
        layer_sizes=(784, 512, 10), input_encoding="poisson", steps_spk=50,
        loader_fn=lambda bs, device, seed: get_loaders("MNIST", bs, "./data", device, seed=seed),
    ),
    "FashionMNIST": DatasetSpec(
        layer_sizes=(784, 512, 10), input_encoding="poisson", steps_spk=50,
        loader_fn=lambda bs, device, seed: get_loaders("FASHIONMNIST", bs, "./data", device, seed=seed),
    ),
    "Caltech": DatasetSpec(
        layer_sizes=(1024, 512, 2), input_encoding="poisson", steps_spk=50,
        loader_fn=lambda bs, device, seed: get_caltech_loaders(bs, "./data", device, seed=seed),
    ),
    "NMNIST": DatasetSpec(
        layer_sizes=(1156, 512, 10), input_encoding="event_direct", steps_spk=25,
        loader_fn=lambda bs, device, seed: get_nmnist_loaders(bs, "./data", device, steps_spk=25, seed=seed),
        batch_preprocess_fn=nmnist_preprocess,
    ),
}


# ─────────────────────────────────────────────────────────────────────────
# 9. STATISTICAL ANALYSIS: HH-PC vs LIF-PC, paired t-test across seeds
# ─────────────────────────────────────────────────────────────────────────

@dataclass
class Cfg:
    poisson_scale: float = 1.0
    current_gain: float = 30.0
    I_bias: float = 2.0
    thr: float = 0.8
    lif_tau: float = 0.3
    pc_activation: str = "relu"
    lr: float = 2e-4
    weight_decay: float = 1e-4
    T_infer_train: int = 100
    T_infer_eval: int = 50
    eta_x: float = 0.05
    epochs: int = 15
    batch_size: int = 128
    eval_seed: int = 1234
    patience: int = 3
    stat_seeds: Tuple[int, ...] = (42, 123, 456, 789, 1011)


def run_statistical_analysis(dataset_key: str, cfg: Cfg, device: str, verbose: bool = True) -> Dict[str, Any]:
    """Trains HH-PC and LIF-PC independently for each seed in cfg.stat_seeds on
    `dataset_key`, evaluates on the held-out test set, and runs a paired
    t-test (HH-PC vs LIF-PC) across seeds on accuracy and F1."""
    spec = DATASETS[dataset_key]
    per_seed: List[Dict[str, float]] = []

    for seed in cfg.stat_seeds:
        print(f"\n{'=' * 60}\n  {dataset_key}  |  seed = {seed}\n{'=' * 60}")
        row: Dict[str, float] = {"seed": seed}

        for tag, neuron_type in (("hh", "hh"), ("lif", "lif")):
            set_seed(seed)
            train_loader, val_loader, test_loader = spec.loader_fn(cfg.batch_size, device, seed)
            model = PCSNNet(
                layer_sizes=list(spec.layer_sizes), device=device, neuron_type=neuron_type,
                current_gain=cfg.current_gain, I_bias=cfg.I_bias, thr=cfg.thr, lif_tau=cfg.lif_tau,
                pc_activation=cfg.pc_activation, lr=cfg.lr, weight_decay=cfg.weight_decay,
                input_encoding=spec.input_encoding, poisson_scale=cfg.poisson_scale,
            ).to(device)

            ckpt = f"best_{dataset_key}_{tag}_seed{seed}.pt"
            train(
                model, train_loader, val_loader, device,
                spec.steps_spk, cfg.T_infer_train, cfg.T_infer_eval, cfg.eta_x,
                cfg.epochs, "pc", cfg.eval_seed, cfg.patience, ckpt,
                verbose=verbose, batch_preprocess_fn=spec.batch_preprocess_fn,
                desc=f"{dataset_key}/{tag.upper()}/seed{seed}",
            )
            test_stats = eval_epoch(
                model, test_loader, device, spec.steps_spk, cfg.T_infer_eval, cfg.eta_x,
                "pc", cfg.eval_seed, spec.batch_preprocess_fn,
            )
            row[f"{tag}_acc"] = test_stats["pc_acc"]
            row[f"{tag}_f1"] = test_stats["pc_f1"]
            row[f"{tag}_energy"] = test_stats["pc_energy"]

        per_seed.append(row)

    def col(key):
        return np.array([r[key] for r in per_seed], dtype=np.float64)

    hh_acc, lif_acc = col("hh_acc"), col("lif_acc")
    hh_f1, lif_f1 = col("hh_f1"), col("lif_f1")
    hh_e, lif_e = col("hh_energy"), col("lif_energy")

    summary = {
        "hh_acc_mean": float(hh_acc.mean()), "hh_acc_std": float(hh_acc.std()),
        "lif_acc_mean": float(lif_acc.mean()), "lif_acc_std": float(lif_acc.std()),
        "hh_f1_mean": float(hh_f1.mean()), "hh_f1_std": float(hh_f1.std()),
        "lif_f1_mean": float(lif_f1.mean()), "lif_f1_std": float(lif_f1.std()),
        "hh_energy_mean": float(hh_e.mean()), "hh_energy_std": float(hh_e.std()),
        "lif_energy_mean": float(lif_e.mean()), "lif_energy_std": float(lif_e.std()),
    }
    t_acc, p_acc = stats.ttest_rel(hh_acc, lif_acc)
    t_f1, p_f1 = stats.ttest_rel(hh_f1, lif_f1)

    n = len(cfg.stat_seeds)
    print(f"\n{'=' * 70}\n  {dataset_key}: HH-PC vs LIF-PC  |  {n} seeds\n{'=' * 70}")
    print(f"{'Seed':<8}{'HH Acc':>9}{'LIF Acc':>9}{'HH F1':>9}{'LIF F1':>9}{'HH Nrg':>10}{'LIF Nrg':>10}")
    print("-" * 65)
    for r in per_seed:
        print(f"{r['seed']:<8}{r['hh_acc']:>9.4f}{r['lif_acc']:>9.4f}"
              f"{r['hh_f1']:>9.4f}{r['lif_f1']:>9.4f}{r['hh_energy']:>10.5f}{r['lif_energy']:>10.5f}")
    print("-" * 65)
    print(f"{'Mean':<8}{summary['hh_acc_mean']:>9.4f}{summary['lif_acc_mean']:>9.4f}"
          f"{summary['hh_f1_mean']:>9.4f}{summary['lif_f1_mean']:>9.4f}"
          f"{summary['hh_energy_mean']:>10.5f}{summary['lif_energy_mean']:>10.5f}")
    print(f"{'Std':<8}{summary['hh_acc_std']:>9.4f}{summary['lif_acc_std']:>9.4f}"
          f"{summary['hh_f1_std']:>9.4f}{summary['lif_f1_std']:>9.4f}"
          f"{summary['hh_energy_std']:>10.5f}{summary['lif_energy_std']:>10.5f}")
    print(f"\n  Paired t-test (HH-PC vs LIF-PC), {n} seeds:")
    print(f"    Accuracy : t = {t_acc:+.4f},  p = {p_acc:.4f}  "
          f"{'*** significant (p<0.05)' if p_acc < 0.05 else '(not significant)'}")
    print(f"    F1 Score : t = {t_f1:+.4f},  p = {p_f1:.4f}  "
          f"{'*** significant (p<0.05)' if p_f1 < 0.05 else '(not significant)'}")
    print(f"{'=' * 70}\n")

    return {
        "per_seed": per_seed,
        "summary": summary,
        "ttest_acc": (float(t_acc), float(p_acc)),
        "ttest_f1": (float(t_f1), float(p_f1)),
    }


In [4]:
device = default_device()
device

'cuda'

## Statistical analysis: HH-PC vs LIF-PC
Trains both variants independently across `cfg.stat_seeds`, evaluates on the held-out test set, and runs a paired t-test on accuracy and F1. One cell per dataset, matching `hhpc_base.py`'s architecture and the reference notebook's stat-analysis structure.

In [5]:
cfg = Cfg()
#mnist_results = run_statistical_analysis("MNIST", cfg, device)

In [6]:
#fashionmnist_results = run_statistical_analysis("FashionMNIST", cfg, device)

In [7]:
#caltech_results = run_statistical_analysis("Caltech", cfg, device)

In [8]:
# N-MNIST needs tonic (see first cell) and uses fewer steps_spk / a shorter
# T_infer by convention for event data; override via a dedicated Cfg if desired.
nmnist_cfg = Cfg()
nmnist_results = run_statistical_analysis("NMNIST", nmnist_cfg, device)


  NMNIST  |  seed = 42


  0%|          | 0/1011893601 [00:00<?, ?it/s]

Extracting ./data/NMNIST/train.zip to ./data/NMNIST


  0%|          | 0/169674850 [00:00<?, ?it/s]

Extracting ./data/NMNIST/test.zip to ./data/NMNIST


...

  [NMNIST/LIF/seed1011] epoch 12 | val_acc=0.9258 | 88.1s


NMNIST/LIF/seed1011 ep13/15: 100%|██████████████████████| 422/422 [01:19<00:00,  5.28it/s, E=0.0001]


  [NMNIST/LIF/seed1011] epoch 13 | val_acc=0.9295 | 88.1s


NMNIST/LIF/seed1011 ep14/15: 100%|██████████████████████| 422/422 [01:19<00:00,  5.30it/s, E=0.0001]


  [NMNIST/LIF/seed1011] epoch 14 | val_acc=0.9310 | 87.8s
  [NMNIST/LIF/seed1011] early stop at epoch 14

  NMNIST: HH-PC vs LIF-PC  |  5 seeds
Seed       HH Acc  LIF Acc    HH F1   LIF F1    HH Nrg   LIF Nrg
-----------------------------------------------------------------
42         0.9305   0.9319   0.9296   0.9309   0.00040   0.00037
123        0.9313   0.9350   0.9304   0.9341   0.00054   0.00056
456        0.9345   0.9333   0.9335   0.9323   0.00045   0.00055
789        0.9375   0.9371   0.9367   0.9363   0.00041   0.00056
1011       0.8450   0.9351   0.8019   0.9342   0.00565   0.00049
-----------------------------------------------------------------
Mean       0.9158   0.9345   0.9064   0.9336   0.00149   0.00051
Std        0.0355   0.0018   0.0523   0.0018   0.00208   0.00007

  Paired t-test (HH-PC vs LIF-PC), 5 seeds:
    Accuracy : t = -1.0479,  p = 0.3538  (not significant)
    F1 Score : t = -1.0319,  p = 0.3604  (not significant)

